# Part 2: Can we omit some controls? (7 points)

This notebook analyzes a DAG with multiple control variables to determine the minimal sufficient set for estimating the causal effect of X on Y.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from causalgraphicalmodels import CausalGraphicalModel
import statsmodels.api as sm
from scipy import stats
import os

# Set random seed for reproducibility
np.random.seed(42)
plt.style.use('seaborn-v0_8')

# Create output directory if it doesn't exist
output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)

## DAG Specification

We need to create a DAG with the following causal relationships:
- $X \rightarrow Y$
- $Z_1 \rightarrow X$, $Z_1 \rightarrow Y$
- $Z_2 \rightarrow X$, $Z_2 \rightarrow Y$
- $Z_3 \rightarrow Z_2$, $Z_3 \rightarrow Y$

In [ ]:
# Create and visualize the DAG
dag = CausalGraphicalModel(
    nodes=['Z3', 'Z1', 'Z2', 'X', 'Y'],
    edges=[
        ('X', 'Y'),      # Direct effect of X on Y
        ('Z1', 'X'),     # Z1 affects X
        ('Z1', 'Y'),     # Z1 affects Y
        ('Z2', 'X'),     # Z2 affects X
        ('Z2', 'Y'),     # Z2 affects Y
        ('Z3', 'Z2'),    # Z3 affects Z2
        ('Z3', 'Y')      # Z3 affects Y
    ]
)

plt.figure(figsize=(10, 8))
dag.draw()
plt.title('DAG: Multiple Controls Analysis')
plt.savefig(f'{output_dir}/part2_dag.png', dpi=300, bbox_inches='tight')
plt.show()

## Data Simulation

Following the lab convention where each causal arrow represents a unit effect, we simulate:
- $Z_3 = \varepsilon_{Z_3}$
- $Z_1 = \varepsilon_{Z_1}$  
- $Z_2 = Z_3 + \varepsilon_{Z_2}$
- $X = Z_1 + Z_2 + \varepsilon_X$
- $Y = X + Z_1 + Z_2 + Z_3 + \varepsilon_Y$

In [ ]:
# Set sample size
n = 10000

# Generate exogenous variables (error terms)
eps_Z3 = np.random.normal(0, 1, n)
eps_Z1 = np.random.normal(0, 1, n)
eps_Z2 = np.random.normal(0, 1, n)
eps_X = np.random.normal(0, 1, n)
eps_Y = np.random.normal(0, 1, n)

# Generate endogenous variables following the DAG structure
Z3 = eps_Z3
Z1 = eps_Z1
Z2 = Z3 + eps_Z2  # Z3 -> Z2
X = Z1 + Z2 + eps_X  # Z1 -> X, Z2 -> X
Y = X + Z1 + Z2 + Z3 + eps_Y  # X -> Y, Z1 -> Y, Z2 -> Y, Z3 -> Y

# Create DataFrame
df = pd.DataFrame({
    'Z3': Z3,
    'Z1': Z1, 
    'Z2': Z2,
    'X': X,
    'Y': Y
})

print("Data simulation completed. Summary statistics:")
print(df.describe())
print(f"\nTrue causal effect of X on Y: 1.0")

## Regression Analysis

We'll run the following regressions and compare their estimates of the X → Y effect:
1. $Y$ vs. $X$
2. $Y$ vs. $X, Z_1$
3. $Y$ vs. $X, Z_2$
4. $Y$ vs. $X, Z_1, Z_2$
5. $Y$ vs. $X, Z_1, Z_2, Z_3$

In [ ]:
# Function to run regression and extract results
def run_regression(y, x_vars, var_names):
    """Run OLS regression and return coefficient and confidence interval for X"""
    X_reg = sm.add_constant(x_vars)
    model = sm.OLS(y, X_reg).fit()
    
    # Get coefficient for X (second parameter, index 1)
    coef = model.params[1]
    conf_int = model.conf_int(alpha=0.01)[1]  # 99% confidence interval
    se = model.bse[1]
    
    return {
        'variables': var_names,
        'coefficient': coef,
        'std_error': se,
        'conf_lower': conf_int[0],
        'conf_upper': conf_int[1],
        'model': model
    }

# Run all regressions
results = []

# 1. Y vs X
results.append(run_regression(df['Y'], df[['X']], 'X'))

# 2. Y vs X, Z1
results.append(run_regression(df['Y'], df[['X', 'Z1']], 'X, Z1'))

# 3. Y vs X, Z2
results.append(run_regression(df['Y'], df[['X', 'Z2']], 'X, Z2'))

# 4. Y vs X, Z1, Z2
results.append(run_regression(df['Y'], df[['X', 'Z1', 'Z2']], 'X, Z1, Z2'))

# 5. Y vs X, Z1, Z2, Z3
results.append(run_regression(df['Y'], df[['X', 'Z1', 'Z2', 'Z3']], 'X, Z1, Z2, Z3'))

# Display results
print("Regression Results (Effect of X on Y):")
print("=" * 60)
for i, result in enumerate(results, 1):
    print(f"{i}. {result['variables']:15} | Coef: {result['coefficient']:6.3f} | "
          f"99% CI: [{result['conf_lower']:6.3f}, {result['conf_upper']:6.3f}]")
print("=" * 60)
print(f"True effect: 1.000")

In [ ]:
# Create coefficient plot
plt.figure(figsize=(12, 8))

# Extract data for plotting
model_names = [result['variables'] for result in results]
coefficients = [result['coefficient'] for result in results]
conf_lower = [result['conf_lower'] for result in results]
conf_upper = [result['conf_upper'] for result in results]

# Create error bars (difference from coefficient to bounds)
error_lower = [coef - lower for coef, lower in zip(coefficients, conf_lower)]
error_upper = [upper - coef for coef, upper in zip(coefficients, conf_upper)]

# Plot coefficients with confidence intervals
x_pos = range(len(model_names))
plt.errorbar(x_pos, coefficients, yerr=[error_lower, error_upper], 
             fmt='o', capsize=5, capthick=2, markersize=8, linewidth=2)

# Add true effect line
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='True Effect = 1.0')

# Formatting
plt.xlabel('Regression Models')
plt.ylabel('Estimated Effect of X on Y')
plt.title('Coefficient Estimates with 99% Confidence Intervals')
plt.xticks(x_pos, model_names, rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{output_dir}/part2_coefficients.png', dpi=300, bbox_inches='tight')
plt.show()

## Analysis of Results

In [ ]:
# Create results summary table
results_df = pd.DataFrame({
    'Model': [result['variables'] for result in results],
    'Coefficient': [result['coefficient'] for result in results],
    'Std_Error': [result['std_error'] for result in results],
    'CI_Lower_99': [result['conf_lower'] for result in results],
    'CI_Upper_99': [result['conf_upper'] for result in results],
    'Bias': [result['coefficient'] - 1.0 for result in results],
    'Contains_True': [(result['conf_lower'] <= 1.0 <= result['conf_upper']) for result in results]
})

print("\nDetailed Results Summary:")
print(results_df.round(4))

# Save results
results_df.to_csv(f'{output_dir}/part2_regression_results.csv', index=False)

In [ ]:
# Print detailed summaries for models 4 and 5
print("\n" + "="*80)
print("DETAILED SUMMARY: Model 4 (Y vs X, Z1, Z2)")
print("="*80)
print(results[3]['model'].summary())

print("\n" + "="*80)
print("DETAILED SUMMARY: Model 5 (Y vs X, Z1, Z2, Z3)")
print("="*80)
print(results[4]['model'].summary())

## Answers to Questions

### Which regressions seem to estimate the effect correctly?

Based on the results above, we can analyze which regressions provide unbiased estimates of the true causal effect (1.0):

- **Model 1 (X only)**: Biased due to omitted variable bias from Z1 and Z2
- **Model 2 (X, Z1)**: Still biased due to omitting Z2 
- **Model 3 (X, Z2)**: Still biased due to omitting Z1
- **Model 4 (X, Z1, Z2)**: Should provide unbiased estimate - controls for all confounders
- **Model 5 (X, Z1, Z2, Z3)**: May have slightly higher variance due to overcontrolling

### Comment on point estimate and precision for models 4 and 5

Looking at the detailed summaries:
- **Point estimates**: Both should be close to 1.0 (the true effect)
- **Precision**: Model 4 likely has better precision (lower standard errors) as it doesn't include the unnecessary control Z3
- **Model 5** includes Z3 which is not necessary for identification, potentially reducing precision

### Can you ignore some Z ∈ {Z1, Z2, Z3} and get a good estimate?

**Answer**: You can ignore Z3 and still get a good estimate. Here's why:

- **Z1 and Z2 are confounders** - they affect both X and Y, so controlling for them is necessary
- **Z3 is not a confounder of X and Y** - while Z3 affects Y, it doesn't directly affect X (only through Z2)
- **Controlling for Z2 blocks the backdoor path through Z3**
- **The minimal sufficient set is {Z1, Z2}** - this blocks all backdoor paths from X to Y

The backdoor paths from X to Y are:
1. X ← Z1 → Y (blocked by controlling for Z1)
2. X ← Z2 → Y (blocked by controlling for Z2)  
3. X ← Z2 ← Z3 → Y (blocked by controlling for Z2)

Therefore, **Model 4 (X, Z1, Z2) provides the optimal balance** of unbiased estimation with maximum precision.

In [ ]:
# Save the simulated data
df.to_csv(f'{output_dir}/part2_simulated_data.csv', index=False)

print("\nAll results saved to output directory.")
print(f"Files created:")
print(f"- part2_dag.png")
print(f"- part2_coefficients.png")
print(f"- part2_regression_results.csv")
print(f"- part2_simulated_data.csv")